[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/myway0101/see-read-explain-dl/blob/main/notebooks/ch02_keras_basics.ipynb)

# 2장 Keras로 신경망 구현하기 — 실습 노트북
『보고, 읽고, 설명하는 딥러닝 — Colab에서 직접 만들며 배우는 딥러닝 응용』

2장 본문에 나오는 코드를 순서대로 실행해 봅니다. 위에서부터 차례로 실행하세요.
- 2.3 XOR을 학습하는 MLP
- 2.4 MNIST 숫자 인식
- 2.5 내 손글씨로 테스트하기
- 2.6 Fashion-MNIST와 검증 세트
- 2.7 콜백: EarlyStopping과 ModelCheckpoint
- 2.8 모델 저장과 불러오기
- 실습: 검증 세트 분리 + 조기 종료

## 준비
필요한 라이브러리를 불러오고, 실행할 때마다 결과가 크게 달라지지 않도록 난수 시드를 고정합니다. (GPU에서는 시드를 고정해도 결과가 조금씩 다를 수 있습니다.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
%config InlineBackend.figure_format = 'retina'

tf.keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__, "/ Keras", keras.__version__)

## 2.3 XOR을 학습하는 MLP
입력 2개, 은닉층 유닛 2개, 출력 유닛 1개인 가장 작은 다층 퍼셉트론입니다.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

In [ ]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(units=2, input_shape=(2,), activation='sigmoid'))  # 은닉층
model.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))                    # 출력층
model.compile(loss='mean_squared_error',
              optimizer=tf.keras.optimizers.SGD(learning_rate=0.3))
model.summary()

10,000에포크 학습합니다(가중치를 40,000번 고칩니다). 진행 상황 출력이 2만 줄이 넘으므로 `verbose=0`으로 출력을 끄고, 끝난 뒤 손실만 확인합니다. (Colab에서 8분 정도 걸립니다.)

In [ ]:
history = model.fit(X, y, batch_size=1, epochs=10000, verbose=0)
print("마지막 손실:", history.history['loss'][-1])
print(model.predict(X))

In [ ]:
plt.plot(history.history['loss'])
plt.xlabel('epoch'); plt.ylabel('loss (MSE)'); plt.title('XOR training loss')
plt.show()

### 실습: XOR을 퍼셉트론 하나로 학습해 보기
은닉층 없이 출력 유닛 하나($y = \mathrm{sigmoid}(w_1x_1 + w_2x_2 + b)$)로 같은 학습을 해 보고, 예측값이 어떻게 나오는지 확인합니다.

In [ ]:
single = tf.keras.models.Sequential()
single.add(tf.keras.layers.Dense(units=1, input_shape=(2,), activation='sigmoid'))
single.compile(loss='mean_squared_error', optimizer=tf.keras.optimizers.SGD(learning_rate=0.3))
h1 = single.fit(X, y, batch_size=1, epochs=2000, verbose=0)
print("마지막 손실:", h1.history['loss'][-1])
print(single.predict(X))

## 2.4 MNIST 숫자 인식

In [ ]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
print(train_images.shape, train_labels[:10], test_images.shape)

In [ ]:
plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(train_images[i], cmap="Greys")
    plt.title(train_labels[i]); plt.axis("off")
plt.show()

### 데이터 전처리
28×28 이미지를 784개 숫자로 펼치고, 0~255 픽셀 값을 0~1로 바꿉니다. 정답은 원-핫 벡터로 바꿉니다.

In [ ]:
train_images = train_images.reshape((60000, 784))
train_images = train_images.astype('float32') / 255.0
test_images = test_images.reshape((10000, 784))
test_images = test_images.astype('float32') / 255.0

train_labels = tf.keras.utils.to_categorical(train_labels)
test_labels = tf.keras.utils.to_categorical(test_labels)
print(train_images.shape, train_labels[0])

In [ ]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(64, activation='relu', input_shape=(784,)))
model.add(tf.keras.layers.Dense(10, activation='sigmoid'))

model.compile(optimizer='rmsprop',
              loss='mse',
              metrics=['accuracy'])

In [ ]:
history = model.fit(train_images, train_labels, epochs=5, batch_size=128)

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print('테스트 정확도:', test_acc)

### 학습 곡선 그리기

In [ ]:
loss = history.history['loss']
acc = history.history['accuracy']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'b', label='Training Loss')
plt.plot(epochs, acc, 'r', label='Accuracy')
plt.legend()
plt.xlabel('epochs')
plt.ylabel('loss/acc')
plt.show()

### 비교해 보기: 소프트맥스 + 크로스 엔트로피
1장에서 배운 다중 분류의 기본 조합으로 바꿔 같은 조건에서 학습해 봅니다.

In [ ]:
model_ce = tf.keras.models.Sequential()
model_ce.add(tf.keras.layers.Dense(64, activation='relu', input_shape=(784,)))
model_ce.add(tf.keras.layers.Dense(10, activation='softmax'))
model_ce.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
model_ce.fit(train_images, train_labels, epochs=5, batch_size=128, verbose=0)
print('테스트 정확도 (softmax + CE):', model_ce.evaluate(test_images, test_labels, verbose=0)[1])

## 2.5 내 손글씨로 테스트하기
종이에 숫자 하나를 굵게 쓰고 사진을 찍어 `test.png`로 올립니다. 아래 셀을 실행하면 파일 선택 창이 열립니다.
파일을 올리지 않고 건너뛰면, 컴퓨터 글꼴로 그린 숫자 그림을 대신 만들어 씁니다.

In [ ]:
import os
try:
    from google.colab import files
    uploaded = files.upload()          # test.png 선택
    if uploaded:
        os.rename(list(uploaded)[0], 'test.png')
except Exception as e:
    print("업로드 건너뜀:", e)

if not os.path.exists('test.png'):
    # 대신 쓸 그림: 흰 종이에 검은 글씨로 '2'를 그린다
    from PIL import Image, ImageDraw, ImageFont
    img = Image.new('L', (200, 200), 255)
    draw = ImageDraw.Draw(img)
    from matplotlib import font_manager
    font = ImageFont.truetype(font_manager.findfont("DejaVu Sans:bold"), 150)
    draw.text((50, 15), "2", fill=0, font=font)
    img.save('test.png')
    print("test.png를 새로 만들었습니다.")

In [ ]:
import cv2 as cv
image = cv.imread('test.png', cv.IMREAD_GRAYSCALE)
image = cv.resize(image, (28, 28))
image = image.astype('float32')
image = image.reshape(1, 784)
image = 255 - image
image /= 255.0
plt.imshow(image.reshape(28, 28), cmap='Greys')
plt.show()

In [ ]:
pred = model.predict(image.reshape(1, 784), batch_size=1)
print("추정된 숫자 =", pred.argmax())
print("출력값:", np.round(pred, 3))

## 2.6 Fashion-MNIST와 검증 세트

In [ ]:
from tensorflow import keras
from tensorflow.keras import datasets, layers, models

fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
print(train_images.shape, train_labels.shape, test_images.shape, test_labels.shape)

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
plt.figure(figsize=(10, 2.4))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(train_images[i], cmap='Greys')
    plt.title(class_names[train_labels[i]], fontsize=7); plt.axis('off')
plt.show()

In [ ]:
train_images = train_images / 255.0
test_images = test_images / 255.0
print(train_labels[0])

In [ ]:
model = models.Sequential()
model.add(layers.Flatten(input_shape=(28, 28)))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_images, train_labels, epochs=5)

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print('정확도:', test_acc)

### 검증 세트를 떼어 두고 학습하기
`validation_split=0.2`를 주면 학습 데이터의 마지막 20%를 검증에 씁니다. 과적합이 어떻게 보이는지 30에포크 동안 관찰합니다.

In [ ]:
model = models.Sequential()
model.add(layers.Flatten(input_shape=(28, 28)))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_images, train_labels, epochs=30, validation_split=0.2, verbose=2)

In [ ]:
h = history.history
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
ep = range(1, len(h['loss']) + 1)
plt.plot(ep, h['loss'], label='train loss'); plt.plot(ep, h['val_loss'], label='validation loss')
plt.xlabel('epoch'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(ep, h['accuracy'], label='train acc'); plt.plot(ep, h['val_accuracy'], label='validation acc')
plt.xlabel('epoch'); plt.legend()
plt.show()
print("검증 손실이 가장 작았던 에포크:", int(np.argmin(h['val_loss'])) + 1)

## 2.7 콜백: EarlyStopping과 ModelCheckpoint

In [ ]:
from keras.callbacks import ModelCheckpoint, EarlyStopping

modelpath = 'best_model.h5'
checkpointer = ModelCheckpoint(filepath=modelpath, monitor='val_loss', verbose=1, save_best_only=True)
early_stopping = EarlyStopping(monitor='val_loss', patience=5)

In [ ]:
model = models.Sequential()
model.add(layers.Flatten(input_shape=(28, 28)))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_images, train_labels, validation_split=0.2, epochs=100, batch_size=200,
                    verbose=0, callbacks=[early_stopping, checkpointer])
print("실제로 학습한 에포크 수:", len(history.history['loss']))

## 2.8 모델 저장과 불러오기
### 모델 구조와 가중치를 함께 저장하기

In [ ]:
model.save('model.h5')

from keras.models import load_model
model2 = load_model('model.h5')
print('불러온 모델의 테스트 정확도:', model2.evaluate(test_images, test_labels, verbose=0)[1])

### 모델 구조만 저장하기 (JSON)

In [ ]:
from keras.models import model_from_json

json_string = model.to_json()
model3 = model_from_json(json_string)
print(json_string[:200], '...')

### 가중치만 저장하기
현재 케라스에서는 가중치 파일 이름이 반드시 `.weights.h5`로 끝나야 합니다.

In [ ]:
model.save_weights('model.weights.h5')

model3.load_weights('model.weights.h5')
model3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('구조(JSON) + 가중치로 되살린 모델의 정확도:', model3.evaluate(test_images, test_labels, verbose=0)[1])

### 체크포인트로 저장된 최고 모델 불러오기

In [ ]:
best = load_model('best_model.h5')
print('검증 손실이 가장 낮았던 모델의 테스트 정확도:', best.evaluate(test_images, test_labels, verbose=0)[1])

## 실습: 검증 세트 분리 + 조기 종료
학습 데이터를 학습용 80%와 검증용 20%로 나누어 최대 100에포크 학습하고, 검증 손실을 기준으로 조기 종료를 적용합니다.
학습이 끝나면 학습·검증 정확도와 손실 그래프를 그리고, 테스트 정확도도 확인합니다.
EarlyStopping의 patience를 2, 5, 10으로 바꾸면 멈추는 시점과 테스트 정확도가 어떻게 달라지는지도 비교해 보세요. 먼저 직접 작성해 본 뒤, 아래 예시 답안과 비교해 보세요.

In [ ]:
# 직접 작성해 보세요



### 예시 답안

In [ ]:
from sklearn.model_selection import train_test_split

x_tr, x_val, y_tr, y_val = train_test_split(train_images, train_labels, test_size=0.2,
                                            random_state=42, stratify=train_labels)

model = models.Sequential()
model.add(layers.Flatten(input_shape=(28, 28)))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(x_tr, y_tr, validation_data=(x_val, y_val), epochs=100,
                    callbacks=[early_stopping], verbose=2)

h = history.history
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
plt.plot(h['accuracy'], label='train'); plt.plot(h['val_accuracy'], label='validation')
plt.title('accuracy'); plt.xlabel('epoch'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(h['loss'], label='train'); plt.plot(h['val_loss'], label='validation')
plt.title('loss'); plt.xlabel('epoch'); plt.legend()
plt.show()
print('테스트 정확도:', model.evaluate(test_images, test_labels, verbose=0)[1])